# SLM-SAM2 Thigh Segmentation — Sheffield Dataset (Lambda)

Runs [mazurowski-lab/SLM-SAM2](https://github.com/mazurowski-lab/SLM-SAM2) on the
69 Sheffield augmented DICOM volumes using **MuscleMap WB Sheffield segmentations
as mask prompts**.

⚠️ **Requires MuscleMap WB Sheffield segmentations** — run
`lambda_musclemap_wb_sheffield.ipynb` first and download results to
`eval_notebooks/muscle_map_wb/sheffield_segs/` before uploading here.

Data: `~/sheffeld/20440164/Aug_N.dcm`
MM WB segs: `~/musclemap_wb_sheffield_segs/Aug_N_dseg.nii.gz`
Output: `~/slmsam_sheffield_segs/Aug_N_slmsam2.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb/sheffield_segs/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/musclemap_wb_sheffield_segs/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/slmsam_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/muscle_map_wb+slmsam/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os

REPO_DIR = '/home/ubuntu/SLM-SAM2'

if not os.path.isdir(REPO_DIR):
    subprocess.check_call(['git', 'clone',
        'https://github.com/mazurowski-lab/SLM-SAM2.git', REPO_DIR])

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--user', '-q',
                        'SimpleITK', 'Pillow', 'pydicom'])
print('Dependencies ready')

In [ ]:
import subprocess, os

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

mem = subprocess.run(['nvidia-smi', '--query-gpu=memory.free,memory.total',
                      '--format=csv,noheader,nounits'],
                     capture_output=True, text=True)
free, total = [int(x) for x in mem.stdout.strip().split(', ')]
print(f'GPU memory: {free} MiB free / {total} MiB total')
if free < 4000:
    print('WARNING: low GPU memory — shut down other kernels first.')

In [ ]:
import urllib.request

CKPT_DIR  = '/home/ubuntu/SLM-SAM2/checkpoints'
CKPT_FILE = os.path.join(CKPT_DIR, 'sam2.1_hiera_tiny.pt')
URL = 'https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_tiny.pt'

os.makedirs(CKPT_DIR, exist_ok=True)
if not os.path.exists(CKPT_FILE):
    print('Downloading checkpoint...')
    urllib.request.urlretrieve(URL, CKPT_FILE)
    print(f'Done ({os.path.getsize(CKPT_FILE) // 1_000_000} MB)')
else:
    print('Checkpoint already present')

In [ ]:
import glob, re, shutil, tempfile
import numpy as np
import torch
import SimpleITK as sitk
import pydicom
from PIL import Image

IMG_DIR    = os.path.expanduser('~/sheffeld/20440164')
MM_SEG_DIR = os.path.expanduser('~/musclemap_wb_sheffield_segs')
OUTPUT_DIR = os.path.expanduser('~/slmsam_sheffield_segs')
CHECKPOINT = '/home/ubuntu/SLM-SAM2/checkpoints/sam2.1_hiera_tiny.pt'
MODEL_CFG  = 'configs/sam2.1/slm_sam2_hiera_t.yaml'
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

# MuscleMap WB 7xxx label scheme — thigh muscles only
LABEL_MAP = {
    7101: 'Vastus_Lateralis_L',    7102: 'Vastus_Lateralis_R',
    7111: 'Vastus_Intermedius_L',  7112: 'Vastus_Intermedius_R',
    7121: 'Vastus_Medialis_L',     7122: 'Vastus_Medialis_R',
    7131: 'Rectus_Femoris_L',      7132: 'Rectus_Femoris_R',
    7141: 'Sartorius_L',           7142: 'Sartorius_R',
    7151: 'Gracilis_L',            7152: 'Gracilis_R',
    7161: 'Semimembranosus_L',     7162: 'Semimembranosus_R',
    7171: 'Semitendinosus_L',      7172: 'Semitendinosus_R',
    7181: 'Biceps_Femoris_L',      7182: 'Biceps_Femoris_R',
    7201: 'Adductor_Magnus_L',     7202: 'Adductor_Magnus_R',
}
MUSCLE_BATCH_SIZE = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)
print(f'Device: {DEVICE}  |  {len(dcm_files)} volumes')
print(f'MM WB segs dir exists: {os.path.isdir(MM_SEG_DIR)}')

In [ ]:
from sam2.build_sam import build_sam2_video_predictor

torch.autocast(device_type='cuda', dtype=torch.bfloat16).__enter__()
if torch.cuda.is_available() and torch.cuda.get_device_properties(0).major >= 8:
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True

predictor = build_sam2_video_predictor(MODEL_CFG, CHECKPOINT)
predictor.eval()
print('SLM-SAM2 predictor loaded on', DEVICE)

In [ ]:
def export_slices_as_jpg(img_array, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    for i in range(img_array.shape[0]):
        sl = img_array[i]
        sl_norm  = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
        sl_uint8 = (sl_norm * 255).astype(np.uint8)
        Image.fromarray(np.stack([sl_uint8] * 3, axis=-1)).save(
            os.path.join(out_dir, f'{i}.jpg')
        )


def find_prompt_slice(muscle_volume):
    slices = np.where(muscle_volume.any(axis=(1, 2)))[0]
    if len(slices) == 0:
        return None
    return int(slices[len(slices) // 2])


print('Helpers defined.')

In [ ]:
# Match volumes to their MM WB segmentations
matched, missing = [], []
for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    seg_path = os.path.join(MM_SEG_DIR, f'Aug_{idx}_dseg.nii.gz')
    if os.path.exists(seg_path):
        matched.append((dcm_path, seg_path, idx))
    else:
        missing.append(f'Aug_{idx}')

print(f'Matched: {len(matched)}  Missing MM seg: {len(missing)}')
if missing:
    print('  Missing:', missing[:10])

In [ ]:
for dcm_path, seg_path, idx in matched:
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_slmsam2.npz')

    if os.path.exists(out_path):
        print(f'Skipping (done): Aug_{idx}')
        continue

    print(f'\nProcessing: Aug_{idx}')
    ds        = pydicom.dcmread(dcm_path)
    img_array = ds.pixel_array.astype(np.float32)   # (D, H, W)
    D, H, W   = img_array.shape
    print(f'  Image shape: {img_array.shape}')

    seg_array = sitk.GetArrayFromImage(sitk.ReadImage(seg_path)).astype(np.int32)

    # Build per-muscle prompts
    muscle_prompts = {}
    for obj_id, (label_idx, muscle_name) in enumerate(LABEL_MAP.items(), start=1):
        muscle_vol   = (seg_array == label_idx)
        prompt_slice = find_prompt_slice(muscle_vol)
        if prompt_slice is None:
            continue
        muscle_prompts[obj_id] = (muscle_name, prompt_slice,
                                   muscle_vol[prompt_slice].astype(np.uint8))

    if not muscle_prompts:
        print('  No muscles found in MM seg — skipping')
        continue

    print(f'  Muscles with prompts: {len(muscle_prompts)}/{len(LABEL_MAP)}')

    tmp_dir   = tempfile.mkdtemp(prefix='slms_')
    all_masks = {}

    try:
        export_slices_as_jpg(img_array, tmp_dir)

        items   = list(muscle_prompts.items())
        batches = [items[i:i+MUSCLE_BATCH_SIZE]
                   for i in range(0, len(items), MUSCLE_BATCH_SIZE)]

        for batch_idx, batch in enumerate(batches):
            print(f'  Batch {batch_idx+1}/{len(batches)}: '
                  f'{[v[0] for _, v in batch]}')

            torch.cuda.empty_cache()
            inference_state = predictor.init_state(video_path=tmp_dir)

            first_prompt = min(v[1] for _, v in batch)

            for obj_id, (muscle_name, prompt_slice, mask_2d) in batch:
                predictor.add_new_mask(
                    inference_state=inference_state,
                    frame_idx=prompt_slice,
                    obj_id=obj_id,
                    mask=mask_2d,
                )

            video_segments = {}
            for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(
                    inference_state, start_frame_idx=first_prompt, recent_n=1):
                video_segments[frame_idx] = {
                    oid: (mask_logits[i] > 0).cpu().numpy()
                    for i, oid in enumerate(obj_ids)
                }

            if first_prompt > 0:
                for frame_idx, obj_ids, mask_logits in predictor.propagate_in_video(
                        inference_state, start_frame_idx=first_prompt,
                        reverse=True, recent_n=1):
                    video_segments[frame_idx] = {
                        oid: (mask_logits[i] > 0).cpu().numpy()
                        for i, oid in enumerate(obj_ids)
                    }

            for obj_id, (muscle_name, _, _) in batch:
                vol_mask = np.zeros((D, H, W), dtype=np.uint8)
                for sl_idx in range(D):
                    seg_frame = video_segments.get(sl_idx, {})
                    if obj_id in seg_frame:
                        vol_mask[sl_idx] = np.squeeze(
                            seg_frame[obj_id]).astype(np.uint8)
                all_masks[muscle_name] = vol_mask

            predictor.reset_state(inference_state)
            torch.cuda.empty_cache()

        np.savez_compressed(out_path, **all_masks)
        print(f'  Saved → {out_path}')

        mm_voxels = {LABEL_MAP[k]: int((seg_array == k).sum()) for k in LABEL_MAP}
        print(f'  {"Muscle":<30} {"MM input":>10} {"Refined":>10}')
        for name, vol in sorted(all_masks.items()):
            rv = int(vol.sum()); mv = mm_voxels.get(name, 0)
            flag = '  <-- EMPTY' if rv == 0 and mv > 0 else ''
            print(f'  {name:<30} {mv:>10,} {rv:>10,}{flag}')

    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Total output files: {len(results)} / {len(matched)}')
if results:
    s = np.load(results[0])
    print(f'Sample: {results[0]}')
    for name in sorted(s.files):
        arr = s[name]
        print(f'  {name:<30} {str(arr.shape):<15} {int(arr.sum()):>10,}')